# Optimizer Comparison — ViT on CIFAR-10

## Cell 1 · Install

In [ ]:
# !pip install -q timm lion-pytorch adabelief-pytorch scikit-learn

## Cell 2 · Imports & Config

In [2]:
import time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

import timm
from lion_pytorch import Lion
from adabelief_pytorch import AdaBelief
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

CFG = dict(
    seed        = 42,
    num_classes = 10,
    img_size    = 224,
    batch_size  = 256,
    num_epochs  = 20,
    num_workers = 8,
    device      = 'cuda' if torch.cuda.is_available() else 'cpu',
)

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print(f"Device : {CFG['device']}")
if CFG['device'] == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


torch.backends.cudnn.benchmark = True


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## Cell 3 · DataLoaders

In [3]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

val_tf = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_ds = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=train_tf)
val_ds   = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

print(f'Train : {len(train_ds):,} samples | {len(train_loader)} batches')
print(f'Val   : {len(val_ds):,} samples  | {len(val_loader)} batches')
print(f'Classes : {train_ds.classes}')


100%|██████████| 170M/170M [1:01:37<00:00, 46.1kB/s] 


Train : 50,000 samples | 196 batches
Val   : 10,000 samples  | 40 batches
Classes : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Cell 4 · Model · Optimizer · Training loop

In [4]:
def get_model(model_name):
    if model_name == 'deit_tiny':
        model = timm.create_model('deit_tiny_patch16_224', pretrained=True,
                                   num_classes=CFG['num_classes'])
    elif model_name == 'resnet18':
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, CFG['num_classes'])

    model = model.to(CFG['device'])
    model = model.to(memory_format=torch.channels_last)

    try:
        model = torch.compile(model)
    except Exception:
        pass

    return model

OPTIMIZER_CONFIGS = {
    'AdamW': dict(
        cls=optim.AdamW,
        kwargs=dict(lr=1e-4, weight_decay=0.05, betas=(0.9, 0.999))
    ),
    'Lion': dict(
        cls=Lion,
        kwargs=dict(lr=1e-5, weight_decay=0.05, betas=(0.9, 0.99))
    ),
    'SGD+Momentum': dict(
        cls=optim.SGD,
        kwargs=dict(lr=1e-2, momentum=0.9, weight_decay=1e-4, nesterov=True)
    ),
    'AdaBelief': dict(
        cls=AdaBelief,
        kwargs=dict(lr=1e-4, eps=1e-16, betas=(0.9, 0.999),
                    weight_decay=1e-4, weight_decouple=True,
                    rectify=False, print_change_log=False)
    ),
    'RMSProp': dict(
        cls=optim.RMSprop,
        kwargs=dict(lr=1e-4, alpha=0.99, weight_decay=1e-4)
    ),
}

def get_optimizer(name, params):
    c = OPTIMIZER_CONFIGS[name]
    return c['cls'](params, **c['kwargs'])

def get_scheduler(opt, n):
    return optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n, eta_min=1e-6)

scaler = torch.cuda.amp.GradScaler(enabled=(CFG['device']=='cuda'))

def compute_grad_norm(model):
    total = sum(p.grad.data.norm(2).item()**2
                for p in model.parameters() if p.grad is not None)
    return total ** 0.5

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total, grad_norms = 0.0, 0, 0, []

    for batch_idx, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(CFG['device'], non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(CFG['device'], non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(CFG['device']=='cuda')):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        if batch_idx % 10 == 0:
            grad_norms.append(compute_grad_norm(model))

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.detach() * imgs.size(0)
        correct += logits.detach().argmax(1).eq(labels).sum().item()
        total += imgs.size(0)

    return total_loss.cpu().item()/total, correct/total, float(np.mean(grad_norms))

@torch.inference_mode()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs   = imgs.to(CFG['device'], non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(CFG['device'], non_blocking=True)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        preds       = logits.argmax(1)
        correct    += preds.eq(labels).sum().item()
        total      += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss/total, correct/total, f1_score(all_labels, all_preds, average='macro')

def run_experiment(model_name, opt_name):
    print(f'\n{"="*60}')
    print(f'  Model: {model_name.upper()}  |  Optimizer: {opt_name}')
    print(f'{"="*60}')
    torch.cuda.empty_cache(); gc.collect()
    if CFG['device'] == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    model     = get_model(model_name)
    optimizer = get_optimizer(opt_name, model.parameters())
    scheduler = get_scheduler(optimizer, CFG['num_epochs'])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    history   = defaultdict(list)
    t_total   = 0.0
    for epoch in range(1, CFG['num_epochs'] + 1):
        t0 = time.time()
        tr_loss, tr_acc, grad_norm = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, va_f1    = evaluate(model, val_loader, criterion)
        scheduler.step()
        et = time.time() - t0; t_total += et
        for k, v in zip(
            ['train_loss','train_acc','val_loss','val_acc','val_f1','grad_norm','epoch_time'],
            [tr_loss, tr_acc, va_loss, va_acc, va_f1, grad_norm, et]
        ):
            history[k].append(v)
        print(f'  Ep {epoch:02d}/{CFG["num_epochs"]} | '
              f'tr={tr_loss:.4f}/{tr_acc:.4f} '
              f'va={va_loss:.4f}/{va_acc:.4f} '
              f'f1={va_f1:.4f} g={grad_norm:.3f} {et:.1f}s')
    peak_mem  = torch.cuda.max_memory_allocated()/1e6 if CFG['device']=='cuda' else 0.0
    epochs_80 = next((i+1 for i, a in enumerate(history['val_acc']) if a >= 0.80), None)
    del model; torch.cuda.empty_cache(); gc.collect()
    return dict(
        model=model_name, optimizer=opt_name,
        best_val_acc=max(history['val_acc']),
        final_val_acc=history['val_acc'][-1],
        best_val_f1=max(history['val_f1']),
        final_val_loss=history['val_loss'][-1],
        total_time_min=t_total/60,
        avg_epoch_s=float(np.mean(history['epoch_time'])),
        peak_memory_mb=peak_mem,
        epochs_to_80=epochs_80,
        history=dict(history),
    )

all_results = []
print('Ready.')


Ready.


## Run 1/10 — DeiT-Tiny + AdamW

In [5]:
r_deit_tiny_adamw = run_experiment('deit_tiny', 'AdamW')
all_results.append(r_deit_tiny_adamw)
print(f'Best val acc : {r_deit_tiny_adamw["best_val_acc"]:.4f}')
print(f'Total time   : {r_deit_tiny_adamw["total_time_min"]:.1f} min')

# ============================================================
#   Model: DEIT_TINY  |  Optimizer: AdamW
# ============================================================
# Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
# model.safetensors: 100%
#  22.9M/22.9M [00:01<00:00, 30.9MB/s]
# W0620 11:18:32.072000 122 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
#   Ep 01/20 | tr=0.8541/0.8621 va=0.6706/0.9282 f1=0.9283 g=253454.247 142.2s
#   Ep 02/20 | tr=0.6169/0.9530 va=0.6147/0.9524 f1=0.9524 g=224015.855 68.2s
#   Ep 03/20 | tr=0.5770/0.9695 va=0.6080/0.9562 f1=0.9563 g=190905.210 67.9s
#   Ep 04/20 | tr=0.5565/0.9784 va=0.6077/0.9555 f1=0.9555 g=192718.773 67.9s
#   Ep 05/20 | tr=0.5397/0.9856 va=0.6066/0.9568 f1=0.9567 g=172763.503 67.7s
#   Ep 06/20 | tr=0.5292/0.9900 va=0.6069/0.9583 f1=0.9584 g=165781.823 68.1s
#   Ep 07/20 | tr=0.5229/0.9919 va=0.6064/0.9578 f1=0.9577 g=163158.445 67.7s
#   Ep 08/20 | tr=0.5166/0.9949 va=0.6017/0.9623 f1=0.9622 g=137375.547 67.8s
#   Ep 09/20 | tr=0.5137/0.9953 va=0.6066/0.9604 f1=0.9604 g=108476.658 67.4s
#   Ep 10/20 | tr=0.5094/0.9972 va=0.6148/0.9590 f1=0.9588 g=76698.547 67.4s
#   Ep 11/20 | tr=0.5054/0.9986 va=0.6023/0.9640 f1=0.9640 g=175856.832 68.1s
#   Ep 12/20 | tr=0.5034/0.9992 va=0.6082/0.9640 f1=0.9641 g=109207.344 67.8s
#   Ep 13/20 | tr=0.5023/0.9995 va=0.6087/0.9640 f1=0.9640 g=66232.278 67.3s
#   Ep 14/20 | tr=0.5018/0.9997 va=0.6112/0.9639 f1=0.9639 g=25450.100 67.3s
#   Ep 15/20 | tr=0.5016/0.9997 va=0.6177/0.9625 f1=0.9625 g=31781.405 67.1s
#   Ep 16/20 | tr=0.5014/0.9998 va=0.6214/0.9620 f1=0.9620 g=26731.951 67.5s
#   Ep 17/20 | tr=0.5012/0.9997 va=0.6222/0.9612 f1=0.9612 g=24234.645 67.0s
#   Ep 18/20 | tr=0.5012/0.9998 va=0.6233/0.9608 f1=0.9608 g=9772.076 67.3s
#   Ep 19/20 | tr=0.5011/0.9998 va=0.6250/0.9607 f1=0.9607 g=41634.507 66.8s
#   Ep 20/20 | tr=0.5010/0.9998 va=0.6250/0.9610 f1=0.9609 g=33036.342 67.0s
# Best val acc : 0.9640
# Total time   : 23.8 min


  Model: DEIT_TINY  |  Optimizer: AdamW


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

W0620 11:18:32.072000 122 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


  Ep 01/20 | tr=0.8541/0.8621 va=0.6706/0.9282 f1=0.9283 g=253454.247 142.2s
  Ep 02/20 | tr=0.6169/0.9530 va=0.6147/0.9524 f1=0.9524 g=224015.855 68.2s
  Ep 03/20 | tr=0.5770/0.9695 va=0.6080/0.9562 f1=0.9563 g=190905.210 67.9s
  Ep 04/20 | tr=0.5565/0.9784 va=0.6077/0.9555 f1=0.9555 g=192718.773 67.9s
  Ep 05/20 | tr=0.5397/0.9856 va=0.6066/0.9568 f1=0.9567 g=172763.503 67.7s
  Ep 06/20 | tr=0.5292/0.9900 va=0.6069/0.9583 f1=0.9584 g=165781.823 68.1s
  Ep 07/20 | tr=0.5229/0.9919 va=0.6064/0.9578 f1=0.9577 g=163158.445 67.7s
  Ep 08/20 | tr=0.5166/0.9949 va=0.6017/0.9623 f1=0.9622 g=137375.547 67.8s
  Ep 09/20 | tr=0.5137/0.9953 va=0.6066/0.9604 f1=0.9604 g=108476.658 67.4s
  Ep 10/20 | tr=0.5094/0.9972 va=0.6148/0.9590 f1=0.9588 g=76698.547 67.4s
  Ep 11/20 | tr=0.5054/0.9986 va=0.6023/0.9640 f1=0.9640 g=175856.832 68.1s
  Ep 12/20 | tr=0.5034/0.9992 va=0.6082/0.9640 f1=0.9641 g=109207.344 67.8s
  Ep 13/20 | tr=0.5023/0.9995 va=0.6087/0.9640 f1=0.9640 g=66232.278 67.3s
  Ep 14/20 | 

## Run 2/10 — DeiT-Tiny + Lion

In [6]:
r_deit_tiny_lion = run_experiment('deit_tiny', 'Lion')
all_results.append(r_deit_tiny_lion)
print(f'Best val acc : {r_deit_tiny_lion["best_val_acc"]:.4f}')
print(f'Total time   : {r_deit_tiny_lion["total_time_min"]:.1f} min')


# ============================================================
#   Model: DEIT_TINY  |  Optimizer: Lion
# ============================================================
#   Ep 01/20 | tr=1.1750/0.7318 va=0.7559/0.8985 f1=0.8984 g=761445.998 71.5s
#   Ep 02/20 | tr=0.6710/0.9314 va=0.6360/0.9442 f1=0.9442 g=802205.312 71.5s
#   Ep 03/20 | tr=0.5858/0.9664 va=0.6123/0.9518 f1=0.9518 g=729288.210 71.0s
#   Ep 04/20 | tr=0.5462/0.9828 va=0.6089/0.9556 f1=0.9557 g=589262.273 70.9s
#   Ep 05/20 | tr=0.5282/0.9894 va=0.6183/0.9528 f1=0.9527 g=278163.522 71.2s
#   Ep 06/20 | tr=0.5215/0.9921 va=0.6104/0.9596 f1=0.9596 g=342426.368 71.0s
#   Ep 07/20 | tr=0.5180/0.9933 va=0.6196/0.9563 f1=0.9563 g=300224.152 71.3s
#   Ep 08/20 | tr=0.5141/0.9943 va=0.6214/0.9576 f1=0.9576 g=349555.024 71.0s
#   Ep 09/20 | tr=0.5108/0.9960 va=0.6224/0.9584 f1=0.9584 g=300877.496 70.4s
#   Ep 10/20 | tr=0.5097/0.9964 va=0.6289/0.9590 f1=0.9589 g=265424.736 70.5s
#   Ep 11/20 | tr=0.5071/0.9975 va=0.6366/0.9583 f1=0.9583 g=98238.406 70.0s
#   Ep 12/20 | tr=0.5063/0.9974 va=0.6369/0.9590 f1=0.9590 g=111136.097 70.3s
#   Ep 13/20 | tr=0.5055/0.9981 va=0.6365/0.9604 f1=0.9604 g=129499.736 70.4s
#   Ep 14/20 | tr=0.5041/0.9987 va=0.6376/0.9604 f1=0.9604 g=169148.143 70.0s
#   Ep 15/20 | tr=0.5034/0.9989 va=0.6320/0.9637 f1=0.9637 g=82498.131 70.0s
#   Ep 16/20 | tr=0.5020/0.9994 va=0.6324/0.9624 f1=0.9624 g=38988.974 69.9s
#   Ep 17/20 | tr=0.5014/0.9996 va=0.6368/0.9623 f1=0.9623 g=108399.894 69.8s
#   Ep 18/20 | tr=0.5012/0.9997 va=0.6338/0.9629 f1=0.9628 g=27240.231 69.4s
#   Ep 19/20 | tr=0.5013/0.9997 va=0.6374/0.9623 f1=0.9623 g=32537.759 69.5s
#   Ep 20/20 | tr=0.5010/0.9998 va=0.6344/0.9624 f1=0.9624 g=2675.228 69.4s
# Best val acc : 0.9637
# Total time   : 23.5 min


  Model: DEIT_TINY  |  Optimizer: Lion
  Ep 01/20 | tr=1.1750/0.7318 va=0.7559/0.8985 f1=0.8984 g=761445.998 71.5s
  Ep 02/20 | tr=0.6710/0.9314 va=0.6360/0.9442 f1=0.9442 g=802205.312 71.5s
  Ep 03/20 | tr=0.5858/0.9664 va=0.6123/0.9518 f1=0.9518 g=729288.210 71.0s
  Ep 04/20 | tr=0.5462/0.9828 va=0.6089/0.9556 f1=0.9557 g=589262.273 70.9s
  Ep 05/20 | tr=0.5282/0.9894 va=0.6183/0.9528 f1=0.9527 g=278163.522 71.2s
  Ep 06/20 | tr=0.5215/0.9921 va=0.6104/0.9596 f1=0.9596 g=342426.368 71.0s
  Ep 07/20 | tr=0.5180/0.9933 va=0.6196/0.9563 f1=0.9563 g=300224.152 71.3s
  Ep 08/20 | tr=0.5141/0.9943 va=0.6214/0.9576 f1=0.9576 g=349555.024 71.0s
  Ep 09/20 | tr=0.5108/0.9960 va=0.6224/0.9584 f1=0.9584 g=300877.496 70.4s
  Ep 10/20 | tr=0.5097/0.9964 va=0.6289/0.9590 f1=0.9589 g=265424.736 70.5s
  Ep 11/20 | tr=0.5071/0.9975 va=0.6366/0.9583 f1=0.9583 g=98238.406 70.0s
  Ep 12/20 | tr=0.5063/0.9974 va=0.6369/0.9590 f1=0.9590 g=111136.097 70.3s
  Ep 13/20 | tr=0.5055/0.9981 va=0.6365/0.9604 f1

## Run 3/10 — DeiT-Tiny + SGD+Momentum

In [7]:
r_deit_tiny_sgd_momentum = run_experiment('deit_tiny', 'SGD+Momentum')
all_results.append(r_deit_tiny_sgd_momentum)
print(f'Best val acc : {r_deit_tiny_sgd_momentum["best_val_acc"]:.4f}')
print(f'Total time   : {r_deit_tiny_sgd_momentum["total_time_min"]:.1f} min')


# ============================================================
#   Model: DEIT_TINY  |  Optimizer: SGD+Momentum
# ============================================================
#   Ep 01/20 | tr=2.4475/0.0765 va=2.4435/0.0756 f1=0.0764 g=589754.103 68.3s
#   Ep 02/20 | tr=2.4460/0.0759 va=2.4417/0.0763 f1=0.0772 g=644248.083 68.2s
#   Ep 03/20 | tr=2.4440/0.0761 va=2.4398/0.0760 f1=0.0769 g=647146.947 68.4s
#   Ep 04/20 | tr=2.4423/0.0759 va=2.4380/0.0768 f1=0.0775 g=640970.950 67.7s
#   Ep 05/20 | tr=2.4403/0.0768 va=2.4361/0.0766 f1=0.0772 g=643708.887 69.0s
#   Ep 06/20 | tr=2.4385/0.0766 va=2.4344/0.0767 f1=0.0773 g=638302.750 68.0s
#   Ep 07/20 | tr=2.4366/0.0769 va=2.4327/0.0759 f1=0.0766 g=634054.292 68.0s
#   Ep 08/20 | tr=2.4351/0.0762 va=2.4312/0.0760 f1=0.0767 g=630453.827 67.9s
#   Ep 09/20 | tr=2.4337/0.0770 va=2.4298/0.0759 f1=0.0767 g=633385.778 68.2s
#   Ep 10/20 | tr=2.4323/0.0765 va=2.4285/0.0753 f1=0.0760 g=631085.594 68.3s
#   Ep 11/20 | tr=2.4311/0.0765 va=2.4275/0.0751 f1=0.0760 g=627649.685 68.0s
#   Ep 12/20 | tr=2.4305/0.0760 va=2.4265/0.0748 f1=0.0759 g=623285.935 67.9s
#   Ep 13/20 | tr=2.4296/0.0763 va=2.4258/0.0751 f1=0.0763 g=619884.640 68.1s
#   Ep 14/20 | tr=2.4286/0.0765 va=2.4252/0.0747 f1=0.0761 g=617474.339 68.1s
#   Ep 15/20 | tr=2.4282/0.0760 va=2.4247/0.0747 f1=0.0760 g=616959.518 68.9s
#   Ep 16/20 | tr=2.4276/0.0756 va=2.4244/0.0747 f1=0.0759 g=623810.159 68.4s
#   Ep 17/20 | tr=2.4275/0.0767 va=2.4242/0.0746 f1=0.0759 g=631889.956 68.0s
#   Ep 18/20 | tr=2.4275/0.0756 va=2.4241/0.0745 f1=0.0758 g=632915.430 68.4s
#   Ep 19/20 | tr=2.4271/0.0767 va=2.4240/0.0746 f1=0.0759 g=622870.378 68.4s
#   Ep 20/20 | tr=2.4271/0.0759 va=2.4240/0.0746 f1=0.0759 g=311708.835 68.0s
# Best val acc : 0.0768
# Total time   : 22.7 min


  Model: DEIT_TINY  |  Optimizer: SGD+Momentum
  Ep 01/20 | tr=2.4475/0.0765 va=2.4435/0.0756 f1=0.0764 g=589754.103 68.3s
  Ep 02/20 | tr=2.4460/0.0759 va=2.4417/0.0763 f1=0.0772 g=644248.083 68.2s
  Ep 03/20 | tr=2.4440/0.0761 va=2.4398/0.0760 f1=0.0769 g=647146.947 68.4s
  Ep 04/20 | tr=2.4423/0.0759 va=2.4380/0.0768 f1=0.0775 g=640970.950 67.7s
  Ep 05/20 | tr=2.4403/0.0768 va=2.4361/0.0766 f1=0.0772 g=643708.887 69.0s
  Ep 06/20 | tr=2.4385/0.0766 va=2.4344/0.0767 f1=0.0773 g=638302.750 68.0s
  Ep 07/20 | tr=2.4366/0.0769 va=2.4327/0.0759 f1=0.0766 g=634054.292 68.0s
  Ep 08/20 | tr=2.4351/0.0762 va=2.4312/0.0760 f1=0.0767 g=630453.827 67.9s
  Ep 09/20 | tr=2.4337/0.0770 va=2.4298/0.0759 f1=0.0767 g=633385.778 68.2s
  Ep 10/20 | tr=2.4323/0.0765 va=2.4285/0.0753 f1=0.0760 g=631085.594 68.3s
  Ep 11/20 | tr=2.4311/0.0765 va=2.4275/0.0751 f1=0.0760 g=627649.685 68.0s
  Ep 12/20 | tr=2.4305/0.0760 va=2.4265/0.0748 f1=0.0759 g=623285.935 67.9s
  Ep 13/20 | tr=2.4296/0.0763 va=2.4258/

## Run 4/10 — DeiT-Tiny + AdaBelief

In [1]:
# r_deit_tiny_adabelief = run_experiment('deit_tiny', 'AdaBelief')
# all_results.append(r_deit_tiny_adabelief)
# print(f'Best val acc : {r_deit_tiny_adabelief["best_val_acc"]:.4f}')
# print(f'Total time   : {r_deit_tiny_adabelief["total_time_min"]:.1f} min')

# ============================================================
#   Model: DEIT_TINY  |  Optimizer: AdaBelief
# ============================================================
# model.safetensors: 100%
#  22.9M/22.9M [00:01<00:00, 22.1MB/s]
# Weight decoupling enabled in AdaBelief
# W0620 12:01:55.195000 440 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode
#   Ep 01/20 | tr=1.1553/0.7540 va=0.7571/0.9058 f1=0.9056 g=425070.936 255.7s
#   Ep 02/20 | tr=0.7539/0.8992 va=0.6866/0.9293 f1=0.9293 g=491405.436 172.8s
#   Ep 03/20 | tr=0.7085/0.9161 va=0.6620/0.9368 f1=0.9366 g=492064.546 172.5s
#   Ep 04/20 | tr=0.6821/0.9271 va=0.6502/0.9402 f1=0.9401 g=478731.964 171.0s
#   Ep 05/20 | tr=0.6693/0.9326 va=0.6385/0.9463 f1=0.9463 g=488778.639 173.1s
#   Ep 06/20 | tr=0.6579/0.9367 va=0.6317/0.9483 f1=0.9482 g=inf 175.5s
#   Ep 07/20 | tr=0.6498/0.9389 va=0.6291/0.9474 f1=0.9474 g=483272.602 172.1s
#   Ep 08/20 | tr=0.6415/0.9423 va=0.6246/0.9509 f1=0.9509 g=inf 174.4s
#   Ep 09/20 | tr=0.6344/0.9453 va=0.6224/0.9494 f1=0.9493 g=475136.068 175.0s
#   Ep 10/20 | tr=0.6313/0.9471 va=0.6185/0.9521 f1=0.9520 g=452332.873 172.0s
#   Ep 11/20 | tr=0.6278/0.9494 va=0.6192/0.9528 f1=0.9528 g=239165.327 173.7s
#   Ep 12/20 | tr=0.6216/0.9506 va=0.6134/0.9543 f1=0.9542 g=235615.492 173.1s
#   Ep 13/20 | tr=0.6143/0.9544 va=0.6102/0.9557 f1=0.9556 g=327433.311 170.7s
#   Ep 14/20 | tr=0.6148/0.9543 va=0.6104/0.9556 f1=0.9555 g=357262.126 175.7s
#   Ep 15/20 | tr=0.6110/0.9553 va=0.6093/0.9565 f1=0.9565 g=231699.284 174.9s
#   Ep 16/20 | tr=0.6081/0.9569 va=0.6081/0.9566 f1=0.9565 g=219493.938 174.0s
#   Ep 17/20 | tr=0.6059/0.9574 va=0.6075/0.9562 f1=0.9561 g=406000.928 172.3s
#   Ep 18/20 | tr=0.6052/0.9589 va=0.6072/0.9562 f1=0.9562 g=444928.193 172.4s
#   Ep 19/20 | tr=0.6046/0.9586 va=0.6069/0.9563 f1=0.9562 g=485164.999 170.9s
#   Ep 20/20 | tr=0.6059/0.9575 va=0.6069/0.9568 f1=0.9568 g=457504.017 173.8s
# Best val acc : 0.9568
# Total time   : 59.1 min

## Run 5/10 — DeiT-Tiny + RMSProp

In [16]:
r_deit_tiny_rmsprop = run_experiment('deit_tiny', 'RMSProp')
all_results.append(r_deit_tiny_rmsprop)
print(f'Best val acc : {r_deit_tiny_rmsprop["best_val_acc"]:.4f}')
print(f'Total time   : {r_deit_tiny_rmsprop["total_time_min"]:.1f} min')


# ============================================================
#   Model: DEIT_TINY  |  Optimizer: RMSProp
# ============================================================
#   Ep 01/20 | tr=2.2940/0.1314 va=2.3042/0.1116 f1=0.0352 g=inf 145.0s
#   Ep 02/20 | tr=2.3036/0.0993 va=2.3029/0.1000 f1=0.0182 g=66148.765 143.3s
#   Ep 03/20 | tr=2.3034/0.0981 va=2.3026/0.1000 f1=0.0182 g=46405.732 143.1s
#   Ep 04/20 | tr=2.3029/0.0978 va=2.3029/0.1000 f1=0.0182 g=62756.424 143.6s
#   Ep 05/20 | tr=2.3028/0.0980 va=2.3027/0.1000 f1=0.0182 g=51404.994 141.6s
#   Ep 06/20 | tr=2.3028/0.0992 va=2.3026/0.1000 f1=0.0182 g=63889.991 140.8s
#   Ep 07/20 | tr=2.3027/0.0989 va=2.3026/0.1000 f1=0.0182 g=75973.085 140.6s
#   Ep 08/20 | tr=2.3027/0.1005 va=2.3026/0.1000 f1=0.0182 g=66464.681 141.1s
#   Ep 09/20 | tr=2.3026/0.0989 va=2.3026/0.1000 f1=0.0182 g=117351.214 140.4s
#   Ep 10/20 | tr=2.3026/0.1022 va=2.3026/0.1000 f1=0.0182 g=109324.013 141.0s
#   Ep 11/20 | tr=2.3026/0.0995 va=2.3026/0.1000 f1=0.0182 g=100576.399 140.3s
#   Ep 12/20 | tr=2.3026/0.0987 va=2.3026/0.1000 f1=0.0182 g=49025.779 140.9s
#   Ep 13/20 | tr=2.3026/0.0972 va=2.3026/0.1000 f1=0.0182 g=43755.602 141.1s
#   Ep 14/20 | tr=2.3026/0.0965 va=2.3026/0.1000 f1=0.0182 g=61885.574 140.8s
#   Ep 15/20 | tr=2.3026/0.0987 va=2.3026/0.1000 f1=0.0182 g=84894.189 141.4s
#   Ep 16/20 | tr=2.3026/0.0982 va=2.3026/0.1000 f1=0.0182 g=79788.511 142.0s
#   Ep 17/20 | tr=2.3026/0.0970 va=2.3026/0.1000 f1=0.0182 g=78753.749 142.2s
#   Ep 18/20 | tr=2.3026/0.0981 va=2.3026/0.1000 f1=0.0182 g=76629.017 141.2s
#   Ep 19/20 | tr=2.3027/0.0970 va=2.3026/0.1000 f1=0.0182 g=78877.211 141.1s
#   Ep 20/20 | tr=2.3027/0.0986 va=2.3026/0.1000 f1=0.0182 g=75314.003 140.1s
# Best val acc : 0.1116
# Total time   : 47.2 min


  Model: DEIT_TINY  |  Optimizer: RMSProp
  Ep 01/20 | tr=2.2940/0.1314 va=2.3042/0.1116 f1=0.0352 g=inf 145.0s
  Ep 02/20 | tr=2.3036/0.0993 va=2.3029/0.1000 f1=0.0182 g=66148.765 143.3s
  Ep 03/20 | tr=2.3034/0.0981 va=2.3026/0.1000 f1=0.0182 g=46405.732 143.1s
  Ep 04/20 | tr=2.3029/0.0978 va=2.3029/0.1000 f1=0.0182 g=62756.424 143.6s
  Ep 05/20 | tr=2.3028/0.0980 va=2.3027/0.1000 f1=0.0182 g=51404.994 141.6s
  Ep 06/20 | tr=2.3028/0.0992 va=2.3026/0.1000 f1=0.0182 g=63889.991 140.8s
  Ep 07/20 | tr=2.3027/0.0989 va=2.3026/0.1000 f1=0.0182 g=75973.085 140.6s
  Ep 08/20 | tr=2.3027/0.1005 va=2.3026/0.1000 f1=0.0182 g=66464.681 141.1s
  Ep 09/20 | tr=2.3026/0.0989 va=2.3026/0.1000 f1=0.0182 g=117351.214 140.4s
  Ep 10/20 | tr=2.3026/0.1022 va=2.3026/0.1000 f1=0.0182 g=109324.013 141.0s
  Ep 11/20 | tr=2.3026/0.0995 va=2.3026/0.1000 f1=0.0182 g=100576.399 140.3s
  Ep 12/20 | tr=2.3026/0.0987 va=2.3026/0.1000 f1=0.0182 g=49025.779 140.9s
  Ep 13/20 | tr=2.3026/0.0972 va=2.3026/0.1000 f

## Run 6/10 — ResNet-18 + AdamW

In [14]:
r_resnet18_adamw = run_experiment('resnet18', 'AdamW')
all_results.append(r_resnet18_adamw)
print(f'Best val acc : {r_resnet18_adamw["best_val_acc"]:.4f}')
print(f'Total time   : {r_resnet18_adamw["total_time_min"]:.1f} min')

#   Model: RESNET18  |  Optimizer: AdamW
# ============================================================
#   Ep 01/20 | tr=0.8968/0.8527 va=0.7091/0.9289 f1=0.9288 g=269125.628 203.6s
#   Ep 02/20 | tr=0.6725/0.9478 va=0.6655/0.9477 f1=0.9476 g=183248.115 140.4s
#   Ep 03/20 | tr=0.6192/0.9704 va=0.6471/0.9548 f1=0.9547 g=286094.839 139.6s
#   Ep 04/20 | tr=0.5952/0.9809 va=0.6464/0.9526 f1=0.9524 g=279424.196 139.7s
#   Ep 05/20 | tr=0.5795/0.9871 va=0.6334/0.9568 f1=0.9567 g=317300.794 140.6s
#   Ep 06/20 | tr=0.5663/0.9913 va=0.6315/0.9564 f1=0.9564 g=461005.849 140.3s
#   Ep 07/20 | tr=0.5591/0.9939 va=0.6300/0.9570 f1=0.9570 g=384059.456 139.9s
#   Ep 08/20 | tr=0.5528/0.9957 va=0.6285/0.9578 f1=0.9577 g=631966.071 153.0s
#   Ep 09/20 | tr=0.5497/0.9962 va=0.6296/0.9579 f1=0.9578 g=560158.564 148.9s
#   Ep 10/20 | tr=0.5473/0.9968 va=0.6255/0.9588 f1=0.9588 g=351207.282 139.5s
#   Ep 11/20 | tr=0.5436/0.9976 va=0.6255/0.9565 f1=0.9565 g=342875.963 137.2s
#   Ep 12/20 | tr=0.5407/0.9981 va=0.6241/0.9577 f1=0.9576 g=583310.834 139.5s
#   Ep 13/20 | tr=0.5384/0.9986 va=0.6244/0.9581 f1=0.9580 g=310971.512 138.8s
#   Ep 14/20 | tr=0.5365/0.9989 va=0.6243/0.9573 f1=0.9573 g=277461.941 138.2s
#   Ep 15/20 | tr=0.5352/0.9991 va=0.6232/0.9571 f1=0.9570 g=403401.363 139.2s
#   Ep 16/20 | tr=0.5340/0.9992 va=0.6233/0.9578 f1=0.9578 g=275442.038 141.0s
#   Ep 17/20 | tr=0.5333/0.9995 va=0.6232/0.9576 f1=0.9576 g=277640.076 141.5s
#   Ep 18/20 | tr=0.5329/0.9991 va=0.6233/0.9576 f1=0.9575 g=396721.702 140.5s
#   Ep 19/20 | tr=0.5325/0.9993 va=0.6230/0.9577 f1=0.9577 g=518663.907 140.5s
#   Ep 20/20 | tr=0.5323/0.9994 va=0.6231/0.9573 f1=0.9572 g=537251.046 142.8s
# Best val acc : 0.9588
# Total time   : 48.1 min


  Model: RESNET18  |  Optimizer: AdamW
  Ep 01/20 | tr=0.8968/0.8527 va=0.7091/0.9289 f1=0.9288 g=269125.628 203.6s
  Ep 02/20 | tr=0.6725/0.9478 va=0.6655/0.9477 f1=0.9476 g=183248.115 140.4s
  Ep 03/20 | tr=0.6192/0.9704 va=0.6471/0.9548 f1=0.9547 g=286094.839 139.6s
  Ep 04/20 | tr=0.5952/0.9809 va=0.6464/0.9526 f1=0.9524 g=279424.196 139.7s
  Ep 05/20 | tr=0.5795/0.9871 va=0.6334/0.9568 f1=0.9567 g=317300.794 140.6s
  Ep 06/20 | tr=0.5663/0.9913 va=0.6315/0.9564 f1=0.9564 g=461005.849 140.3s
  Ep 07/20 | tr=0.5591/0.9939 va=0.6300/0.9570 f1=0.9570 g=384059.456 139.9s
  Ep 08/20 | tr=0.5528/0.9957 va=0.6285/0.9578 f1=0.9577 g=631966.071 153.0s
  Ep 09/20 | tr=0.5497/0.9962 va=0.6296/0.9579 f1=0.9578 g=560158.564 148.9s
  Ep 10/20 | tr=0.5473/0.9968 va=0.6255/0.9588 f1=0.9588 g=351207.282 139.5s
  Ep 11/20 | tr=0.5436/0.9976 va=0.6255/0.9565 f1=0.9565 g=342875.963 137.2s
  Ep 12/20 | tr=0.5407/0.9981 va=0.6241/0.9577 f1=0.9576 g=583310.834 139.5s
  Ep 13/20 | tr=0.5384/0.9986 va=0.6

## Run 7/10 — ResNet-18 + Lion

In [15]:
r_resnet18_lion = run_experiment('resnet18', 'Lion')
all_results.append(r_resnet18_lion)
print(f'Best val acc : {r_resnet18_lion["best_val_acc"]:.4f}')
print(f'Total time   : {r_resnet18_lion["total_time_min"]:.1f} min')


# ============================================================
#   Model: RESNET18  |  Optimizer: Lion
# ============================================================
#   Ep 01/20 | tr=0.9216/0.8379 va=0.6893/0.9375 f1=0.9373 g=nan 141.5s
#   Ep 02/20 | tr=0.6408/0.9589 va=0.6469/0.9505 f1=0.9504 g=172676.076 143.1s
#   Ep 03/20 | tr=0.5891/0.9788 va=0.6299/0.9561 f1=0.9560 g=199599.646 142.0s
#   Ep 04/20 | tr=0.5575/0.9891 va=0.6210/0.9567 f1=0.9566 g=188834.430 141.6s
#   Ep 05/20 | tr=0.5397/0.9941 va=0.6138/0.9577 f1=0.9575 g=176511.233 141.7s
#   Ep 06/20 | tr=0.5297/0.9961 va=0.6093/0.9580 f1=0.9580 g=278098.479 141.5s
#   Ep 07/20 | tr=0.5207/0.9976 va=0.6072/0.9565 f1=0.9565 g=238217.414 141.4s
#   Ep 08/20 | tr=0.5161/0.9986 va=0.6058/0.9589 f1=0.9588 g=250631.512 141.6s
#   Ep 09/20 | tr=0.5120/0.9990 va=0.6020/0.9591 f1=0.9591 g=154119.605 141.8s
#   Ep 10/20 | tr=0.5088/0.9993 va=0.5991/0.9592 f1=0.9592 g=132299.599 141.3s
#   Ep 11/20 | tr=0.5075/0.9994 va=0.6013/0.9614 f1=0.9613 g=194608.334 141.4s
#   Ep 12/20 | tr=0.5052/0.9997 va=0.6011/0.9607 f1=0.9606 g=110452.148 142.0s
#   Ep 13/20 | tr=0.5042/0.9998 va=0.5979/0.9602 f1=0.9602 g=91370.235 141.6s
#   Ep 14/20 | tr=0.5032/0.9999 va=0.6013/0.9596 f1=0.9597 g=85369.729 141.5s
#   Ep 15/20 | tr=0.5025/1.0000 va=0.5971/0.9614 f1=0.9614 g=93121.790 141.4s
#   Ep 16/20 | tr=0.5021/1.0000 va=0.5968/0.9615 f1=0.9614 g=76491.494 141.3s
#   Ep 17/20 | tr=0.5018/1.0000 va=0.5970/0.9615 f1=0.9615 g=96940.078 140.9s
#   Ep 18/20 | tr=0.5017/1.0000 va=0.5971/0.9616 f1=0.9615 g=117378.584 141.0s
#   Ep 19/20 | tr=0.5015/1.0000 va=0.5971/0.9620 f1=0.9620 g=110435.371 141.0s
#   Ep 20/20 | tr=0.5013/1.0000 va=0.5984/0.9630 f1=0.9629 g=95504.270 141.4s
# Best val acc : 0.9630
# Total time   : 47.2 min


  Model: RESNET18  |  Optimizer: Lion
  Ep 01/20 | tr=0.9216/0.8379 va=0.6893/0.9375 f1=0.9373 g=nan 141.5s
  Ep 02/20 | tr=0.6408/0.9589 va=0.6469/0.9505 f1=0.9504 g=172676.076 143.1s
  Ep 03/20 | tr=0.5891/0.9788 va=0.6299/0.9561 f1=0.9560 g=199599.646 142.0s
  Ep 04/20 | tr=0.5575/0.9891 va=0.6210/0.9567 f1=0.9566 g=188834.430 141.6s
  Ep 05/20 | tr=0.5397/0.9941 va=0.6138/0.9577 f1=0.9575 g=176511.233 141.7s
  Ep 06/20 | tr=0.5297/0.9961 va=0.6093/0.9580 f1=0.9580 g=278098.479 141.5s
  Ep 07/20 | tr=0.5207/0.9976 va=0.6072/0.9565 f1=0.9565 g=238217.414 141.4s
  Ep 08/20 | tr=0.5161/0.9986 va=0.6058/0.9589 f1=0.9588 g=250631.512 141.6s
  Ep 09/20 | tr=0.5120/0.9990 va=0.6020/0.9591 f1=0.9591 g=154119.605 141.8s
  Ep 10/20 | tr=0.5088/0.9993 va=0.5991/0.9592 f1=0.9592 g=132299.599 141.3s
  Ep 11/20 | tr=0.5075/0.9994 va=0.6013/0.9614 f1=0.9613 g=194608.334 141.4s
  Ep 12/20 | tr=0.5052/0.9997 va=0.6011/0.9607 f1=0.9606 g=110452.148 142.0s
  Ep 13/20 | tr=0.5042/0.9998 va=0.5979/0.96

## Run 8/10 — ResNet-18 + SGD+Momentum

In [ ]:
# r_resnet18_sgd_momentum = run_experiment('resnet18', 'SGD+Momentum')
# all_results.append(r_resnet18_sgd_momentum)
# print(f'Best val acc : {r_resnet18_sgd_momentum["best_val_acc"]:.4f}')
# print(f'Total time   : {r_resnet18_sgd_momentum["total_time_min"]:.1f} min')


# ============================================================
#   Model: RESNET18  |  Optimizer: SGD+Momentum
# ============================================================
# Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
# 100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s] 
# W0620 02:34:59.507000 124 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
#   Ep 01/20 | tr=2.5209/0.0924 va=2.5279/0.0917 f1=0.0537 g=344659.630 143.5s
#   Ep 02/20 | tr=2.5187/0.0925 va=2.5246/0.0910 f1=0.0536 g=345006.902 67.3s
#   Ep 03/20 | tr=2.5174/0.0911 va=2.5195/0.0906 f1=0.0540 g=345204.445 68.7s
#   Ep 04/20 | tr=2.5146/0.0917 va=2.5201/0.0920 f1=0.0542 g=346842.998 68.5s
#   Ep 05/20 | tr=2.5114/0.0920 va=2.5150/0.0913 f1=0.0541 g=339681.203 69.0s
#   Ep 06/20 | tr=2.5100/0.0922 va=2.5170/0.0907 f1=0.0537 g=339207.457 68.7s
#   Ep 07/20 | tr=2.5085/0.0927 va=2.5144/0.0919 f1=0.0544 g=349308.457 68.3s
#   Ep 08/20 | tr=2.5058/0.0929 va=2.5101/0.0904 f1=0.0534 g=341123.786 68.5s
#   Ep 09/20 | tr=2.5048/0.0931 va=2.5120/0.0917 f1=0.0551 g=339672.649 68.5s
#   Ep 10/20 | tr=2.5040/0.0922 va=2.5099/0.0920 f1=0.0551 g=332416.979 68.8s
#   Ep 11/20 | tr=2.5027/0.0913 va=2.5094/0.0919 f1=0.0547 g=614668.968 68.9s
#   Ep 12/20 | tr=2.5013/0.0927 va=2.5076/0.0912 f1=0.0542 g=668894.933 68.5s
#   Ep 13/20 | tr=2.4997/0.0932 va=2.5071/0.0913 f1=0.0552 g=654002.500 68.8s
#   Ep 14/20 | tr=2.4993/0.0931 va=2.5063/0.0919 f1=0.0557 g=662961.426 68.1s
#   Ep 15/20 | tr=2.4994/0.0940 va=2.5072/0.0917 f1=0.0542 g=665827.503 68.5s
#   Ep 16/20 | tr=2.4987/0.0927 va=2.5067/0.0921 f1=0.0556 g=659193.309 67.6s
#   Ep 17/20 | tr=2.4996/0.0918 va=2.5042/0.0912 f1=0.0542 g=673233.007 68.0s
#   Ep 18/20 | tr=2.4984/0.0919 va=2.5061/0.0916 f1=0.0552 g=661743.323 68.7s
#   Ep 19/20 | tr=2.4988/0.0926 va=2.5041/0.0915 f1=0.0543 g=662259.657 68.8s
#   Ep 20/20 | tr=2.4991/0.0926 va=2.5044/0.0923 f1=0.0556 g=664029.597 68.0s
# Best val acc : 0.0923
# Total time   : 24.1 min

## Run 9/10 — ResNet-18 + AdaBelief

In [ ]:
# r_resnet18_adabelief = run_experiment('resnet18', 'AdaBelief')
# all_results.append(r_resnet18_adabelief)
# print(f'Best val acc : {r_resnet18_adabelief["best_val_acc"]:.4f}')
# print(f'Total time   : {r_resnet18_adabelief["total_time_min"]:.1f} min')


# ============================================================
#   Model: RESNET18  |  Optimizer: AdaBelief
# ============================================================
# Weight decoupling enabled in AdaBelief
#   Ep 01/20 | tr=2.1295/0.2635 va=1.9460/0.4075 f1=0.4030 g=626415.712 69.8s
#   Ep 02/20 | tr=1.7817/0.5159 va=1.6275/0.5965 f1=0.5894 g=452908.959 70.0s
#   Ep 03/20 | tr=1.5060/0.6471 va=1.3982/0.6901 f1=0.6866 g=416713.672 70.1s
#   Ep 04/20 | tr=1.3134/0.7194 va=1.2415/0.7418 f1=0.7394 g=395992.383 70.1s
#   Ep 05/20 | tr=1.1826/0.7622 va=1.1387/0.7743 f1=0.7727 g=370822.092 70.1s
#   Ep 06/20 | tr=1.0956/0.7901 va=1.0662/0.7940 f1=0.7933 g=366252.002 70.2s
#   Ep 07/20 | tr=1.0341/0.8116 va=1.0152/0.8127 f1=0.8122 g=354054.132 70.0s
#   Ep 08/20 | tr=0.9888/0.8292 va=0.9783/0.8251 f1=0.8247 g=340887.586 69.4s
#   Ep 09/20 | tr=0.9562/0.8396 va=0.9528/0.8348 f1=0.8344 g=331392.198 70.0s
#   Ep 10/20 | tr=0.9330/0.8478 va=0.9314/0.8429 f1=0.8426 g=326774.127 69.9s
#   Ep 11/20 | tr=0.9150/0.8556 va=0.9169/0.8487 f1=0.8485 g=377110.875 70.1s
#   Ep 12/20 | tr=0.9012/0.8595 va=0.9056/0.8529 f1=0.8528 g=328628.634 69.7s
#   Ep 13/20 | tr=0.8911/0.8622 va=0.8973/0.8571 f1=0.8570 g=328739.061 69.0s
#   Ep 14/20 | tr=0.8826/0.8656 va=0.8910/0.8588 f1=0.8590 g=317941.068 70.4s
#   Ep 15/20 | tr=0.8765/0.8693 va=0.8850/0.8613 f1=0.8612 g=316977.730 69.8s
#   Ep 16/20 | tr=0.8729/0.8696 va=0.8827/0.8629 f1=0.8627 g=313426.722 70.6s
#   Ep 17/20 | tr=0.8706/0.8708 va=0.8805/0.8627 f1=0.8626 g=320749.812 70.6s
#   Ep 18/20 | tr=0.8669/0.8729 va=0.8787/0.8636 f1=0.8636 g=307341.000 70.4s
#   Ep 19/20 | tr=0.8663/0.8732 va=0.8783/0.8645 f1=0.8644 g=315156.553 70.2s
#   Ep 20/20 | tr=0.8659/0.8737 va=0.8783/0.8636 f1=0.8634 g=306755.998 69.7s
# Best val acc : 0.8645
# Total time   : 23.3 min

## Run 10/10 — ResNet-18 + RMSProp

In [ ]:
# r_resnet18_rmsprop = run_experiment('resnet18', 'RMSProp')
# all_results.append(r_resnet18_rmsprop)
# print(f'Best val acc : {r_resnet18_rmsprop["best_val_acc"]:.4f}')
# print(f'Total time   : {r_resnet18_rmsprop["total_time_min"]:.1f} min')

# ============================================================
#   Model: RESNET18  |  Optimizer: RMSProp
# ============================================================
#   Ep 01/20 | tr=2.3018/0.1399 va=2.2654/0.1887 f1=0.1721 g=431668.454 69.5s
#   Ep 02/20 | tr=2.2719/0.1819 va=2.2913/0.1705 f1=0.1303 g=521060.733 68.8s
#   Ep 03/20 | tr=2.2812/0.1483 va=2.2944/0.2090 f1=0.1318 g=512849.722 68.4s
#   Ep 04/20 | tr=2.2848/0.1595 va=2.2975/0.1635 f1=0.0968 g=489306.636 68.3s
#   Ep 05/20 | tr=2.2793/0.1531 va=2.2423/0.1385 f1=0.0685 g=inf 67.3s
#   Ep 06/20 | tr=2.2001/0.2115 va=2.2470/0.1725 f1=0.0951 g=220976.880 68.5s
#   Ep 07/20 | tr=2.1237/0.2356 va=2.6712/0.1694 f1=0.0738 g=148705.419 68.5s
#   Ep 08/20 | tr=2.1156/0.2411 va=2.1414/0.2232 f1=0.1202 g=180067.649 68.1s
#   Ep 09/20 | tr=2.0529/0.2475 va=2.2118/0.2118 f1=0.0977 g=101678.619 67.1s
#   Ep 10/20 | tr=2.0153/0.2590 va=2.0712/0.2399 f1=0.1175 g=107197.861 67.1s
#   Ep 11/20 | tr=2.0160/0.2631 va=2.1140/0.2432 f1=0.1220 g=111782.548 66.2s
#   Ep 12/20 | tr=2.0208/0.2652 va=2.2456/0.1724 f1=0.0918 g=141624.359 66.0s
#   Ep 13/20 | tr=2.0169/0.2693 va=2.0732/0.2631 f1=0.1394 g=133298.423 65.5s
#   Ep 14/20 | tr=2.0159/0.2707 va=2.1188/0.2635 f1=0.1326 g=140414.672 65.6s
#   Ep 15/20 | tr=2.0147/0.2733 va=2.0306/0.2701 f1=0.1335 g=160026.077 64.5s
#   Ep 16/20 | tr=2.0175/0.2749 va=2.1030/0.2644 f1=0.1417 g=161782.320 64.7s
#   Ep 17/20 | tr=2.0225/0.2772 va=2.0662/0.2780 f1=0.1448 g=160486.714 63.6s
#   Ep 18/20 | tr=2.0214/0.2782 va=2.0406/0.2753 f1=0.1362 g=169655.595 64.1s
#   Ep 19/20 | tr=2.0523/0.2789 va=2.0877/0.2772 f1=0.1446 g=283748.147 64.3s
#   Ep 20/20 | tr=2.0849/0.2776 va=2.0830/0.2773 f1=0.1434 g=283459.225 63.9s
# Best val acc : 0.2780
# Total time   : 22.2 min